# Week 8 Hands-on

# Transformer-based Named Entity Recognition (NER)

In this notebook we explore modern Information Extraction using pretrained Transformer models.

Learning objectives:

- Understand Transformer-based NER
- Run a pretrained BERT model
- Explore contextual understanding
- Convert NER output into structured information
- Compare Transformers with classical ML approaches


In [14]:
from transformers import pipeline, AutoTokenizer, AutoModel
import pandas as pd
import numpy as np
from pprint import pprint

## 1. Loading a Pretrained NER Model

We use a BERT model fine-tuned for token classification.


In [15]:
ner = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("NER model loaded.")

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


NER model loaded.


## 2. First Example

Run NER on a simple sentence.


In [16]:
text = "Amazon opened a new office in Paris."

results = ner(text)
pd.DataFrame(results)

,entity_group,score,word,start,end
0,ORG,0.999034,Amazon,0,6
1,LOC,0.999518,Paris,30,35


### Questions

1. Which entities were found?
2. Which labels were assigned?
3. How confident is the model?


## 3. Multiple Examples

In [17]:
examples = [
    "Barack Obama visited Berlin.",
    "Google acquired DeepMind in London.",
    "Microsoft announced a partnership with OpenAI.",
    "Tesla opened a factory in Germany."
]

for text in examples:
    print("="*60)
    print(text)
    display(pd.DataFrame(ner(text)))

Barack Obama visited Berlin.


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,PER,0.999401,Barack Obama,0,12
1,LOC,0.999743,Berlin,21,27


Google acquired DeepMind in London.


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


...

/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,ORG,0.999044,Microsoft,0,9
1,ORG,0.996356,OpenAI,39,45


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Tesla opened a factory in Germany.


,entity_group,score,word,start,end
0,ORG,0.791478,Tesla,0,5
1,LOC,0.999575,Germany,26,33


## 4. Context Matters

BERT produces contextual representations.
The meaning of a word depends on surrounding words.


In [21]:
example_1 = "Apple released a new iPhone."
example_2 = "Apple grows on trees."

print("Example 1")
display(pd.DataFrame(ner(example_1)))

print("Example 2")
display(pd.DataFrame(ner(example_2)))

Example 1


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,ORG,0.998479,Apple,0,5
1,MISC,0.983715,iPhone,21,27


Example 2


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,MISC,0.389512,Apple,0,5


### Discussion

Why might the same word behave differently in different contexts?


## 5. Ambiguous Entities

In [22]:
examples = [
    "Jordan won the match.",
    "Jordan is located in the Middle East.",
    "Washington announced new regulations.",
    "I visited Washington last year."
]

for text in examples:
    print("="*60)
    print(text)
    display(pd.DataFrame(ner(text)))

Jordan won the match.


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,ORG,0.949114,Jordan,0,6


Jordan is located in the Middle East.


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


...

/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


,entity_group,score,word,start,end
0,LOC,0.999086,Washington,0,10


/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


I visited Washington last year.


,entity_group,score,word,start,end
0,LOC,0.999708,Washington,10,20


## 6. From NER to Information Extraction

NER is often only one component in a larger IE pipeline.


In [23]:
def extract_entities(text):

    entities = {
        "PER": [],
        "ORG": [],
        "LOC": [],
        "MISC": []
    }

    for item in ner(text):
        label = item["entity_group"]

        if label in entities:
            entities[label].append(item["word"])

    return entities

In [24]:
extract_entities(
    "Google opened a new office in Berlin and hired John Smith."
)

/opt/anaconda3/envs/exr/lib/python3.10/site-packages/torch/nn/modules/module.py:1747: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


{'PER': ['John Smith'], 'ORG': ['Google'], 'LOC': ['Berlin'], 'MISC': []}

## 7. News Article Extraction

In [25]:
article = '''
Google announced a new research center in Paris.
John Smith will lead the team.
Microsoft and OpenAI are collaborating on several projects.
'''

extract_entities(article)

{'PER': ['John Smith'],
 'ORG': ['Google', 'Microsoft', 'OpenAI'],
 'LOC': ['Paris'],
 'MISC': []}

## 8. Batch Processing Documents

In [26]:
documents = [
    "Amazon opened a new office in Paris.",
    "Barack Obama visited Berlin.",
    "Microsoft partnered with OpenAI."
]

for doc in documents:
    print(doc)
    pprint(extract_entities(doc))
    print()

Amazon opened a new office in Paris.
{'LOC': ['Paris'], 'MISC': [], 'ORG': ['Amazon'], 'PER': []}

Barack Obama visited Berlin.
{'LOC': ['Berlin'], 'MISC': [], 'ORG': [], 'PER': ['Barack Obama']}

Microsoft partnered with OpenAI.
{'LOC': [], 'MISC': [], 'ORG': ['Microsoft', 'OpenAI'], 'PER': []}



## 9. Why Transformers Work Better

Classical ML:

Text
→ handcrafted features
→ classifier

Transformer:

Text
→ Transformer
→ contextual representations
→ NER prediction


## 10. Mini Exercise

Run NER on your own examples.

Try:

- company names
- cities
- politicians
- ambiguous words

Observe when the model succeeds and when it fails.


In [27]:
my_text = "OpenAI opened a research office in Munich."

pd.DataFrame(ner(my_text))

,entity_group,score,word,start,end
0,ORG,0.998554,OpenAI,0,6
1,LOC,0.998716,Munich,35,41


## 11. Error Analysis

Every model makes mistakes.

Try to find:

- missed entities
- wrong labels
- ambiguous examples
- entities unseen during training


## Reflection Questions

1. Which approach required more feature engineering?
2. Why do Transformers generalize better?
3. Why is context important?
4. What are the limitations of pretrained NER?
5. How could NER be used in an Information Extraction pipeline?
